# VoiceGuard -- train AASIST on Kaggle

Trains the countermeasure this project's detector uses, on a free Kaggle GPU.
Companion to [`docs/training_runbook.md`](https://github.com/ghanshyam1011/voice-clone-detector/blob/main/docs/training_runbook.md)
in the repo -- read that for the full reasoning. Short version:

- **Trains AASIST, not SSL-AASIST.** SSL-AASIST is the architecture most likely to reach
  the project's <15% In-the-Wild EER target per the wider literature, but it has no feature
  cache yet (the 94M-param frozen wav2vec2 front-end reruns every step) and has never
  completed a real training loop in this repo. Training it today would waste Kaggle's
  weekly GPU quota on an unoptimized path. AASIST is simpler, already smoke-tested
  (a 2-epoch local run took dev EER from 34.1% to 26.6%), and the training script is
  fully resumable -- built for exactly Kaggle's per-session time limit.
- **Before running:** in the Kaggle notebook editor, turn on a **GPU accelerator**
  (Settings, right-hand panel -- T4x2 or P100) and **Add Input** an ASVspoof 2019 LA
  dataset (search Kaggle's own dataset listing first; community mirrors of this corpus
  are common, and attaching one costs nothing against your quota, unlike uploading your own).

## You do not need to keep your laptop on

A full run (40 epochs, full ASVspoof19 LA train set) takes well beyond a single Kaggle
GPU session -- sessions are capped at roughly 9-12 hours (check the exact figure Kaggle
currently gives your account under the session's status panel; it has changed before),
against a 30-hours/week quota. Two different things people conflate here:

1. **Does the run need your browser/laptop connected while it executes?** No, if you use
   **Save Version -> Save & Run All (Commit)** (top right) instead of clicking through
   cells interactively. That queues the *entire notebook* as a clean, independent session
   on Kaggle's own servers -- close your laptop, close the tab, it keeps running. Running
   cells one at a time in the live editor (what the two screenshots in this project's chat
   history show) *does* stay tied to that interactive session.
2. **Does one committed run finish a 40-epoch training in one shot?** No -- it will still
   hit the same ~9-12h session cap, whichever mode you used to start it. This is expected
   and already handled below: every committed run checks for a checkpoint from a *previous*
   run and resumes from it, so finishing the full 40 epochs just means repeating
   "attach last output as input, Save Version, wait, repeat" a handful of times -- not
   babysitting one continuous 40+ hour session.

In [ ]:
!nvidia-smi
!ffmpeg -version | head -1  # the augmentation pipeline's codec pass needs this on PATH

## 1. Get the code

Clones the public repo as-is. This pulls the training script and everything it imports
(`voiceguard.models`, `voiceguard.data`, `voiceguard.audio`) -- none of that depends on
anything still uncommitted locally, only on what's already pushed.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/ghanshyam1011/voice-clone-detector.git
%cd voice-clone-detector

In [ ]:
!pip install -e ".[dl]" -q

## 2. Point the repo at your attached dataset

If you haven't already: click **Add Input** (top right) and attach an ASVspoof 2019 LA
dataset before running the next cell.

The cell below searches everything under `/kaggle/input/` for the distinctive
`ASVspoof2019_LA_cm_protocols` folder, so it finds the right path regardless of how the
specific dataset you attached happens to be laid out (community uploads nest things
differently). If it reports nothing found, attach a dataset and re-run this cell --
no need to re-run anything above it.

In [ ]:
from pathlib import Path

hits = list(Path('/kaggle/input').rglob('ASVspoof2019_LA_cm_protocols'))
if not hits:
    print('Could not find an ASVspoof2019 LA dataset under /kaggle/input.')
    print('Add one via "Add Input" (top right), then re-run this cell.')
    print()
    print('Currently attached:')
    for p in sorted(Path('/kaggle/input').iterdir()):
        print(' ', p)
else:
    LA_ROOT = hits[0].parent
    print(f'Found ASVspoof2019 LA data at: {LA_ROOT}')
    if len(hits) > 1:
        print(f'({len(hits)} matches found -- using the first; others: {hits[1:]})')

In [ ]:
import os

target = Path('data/raw/LA/LA')
target.parent.mkdir(parents=True, exist_ok=True)
if target.is_symlink() or target.exists():
    if target.is_symlink():
        target.unlink()
    else:
        raise SystemExit(f'{target} already exists and is not a symlink -- remove it first')
os.symlink(LA_ROOT, target)
print(f'linked {target} -> {target.resolve()}')

# the exact layout scripts/build_manifests.py expects (src/voiceguard/data/manifests.py)
expected = [
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt',
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt',
    'ASVspoof2019_LA_train/flac',
    'ASVspoof2019_LA_dev/flac',
    'ASVspoof2019_LA_eval/flac',
]
print()
ok = True
for rel in expected:
    p = target / rel
    exists = p.exists()
    ok &= exists
    print(('OK      ' if exists else 'MISSING '), p)
if not ok:
    print()
    print('Some expected paths are missing -- your attached dataset likely nests the LA')
    print('folder differently than the official release. List the LA_ROOT above manually')
    print('(!find <LA_ROOT> -maxdepth 2) and adjust, or try a different attached dataset.')

In [ ]:
!python scripts/build_manifests.py

## 3. Resuming a previous run (skip this if it's your first run)

`Save & Run All` always starts from a *clean* `/kaggle/working/` -- nothing from an
earlier committed version carries over automatically. To continue training instead of
starting over: open this notebook's previous version's **Output** tab, confirm
`models/cm/aasist/last.pt` is there, then **Add Input -> Notebooks -> (this notebook) ->
pick that version**, and run the cell below. It searches every attached input for a
`last.pt` file and copies the first one it finds into place; `scripts/train_cm.py` then
resumes from it automatically -- nothing else in this notebook needs to change.

If nothing is found, this is treated as a fresh run (correct for your very first commit).

In [ ]:
import shutil

hits = [p for p in Path('/kaggle/input').rglob('last.pt') if 'aasist' in str(p).lower()]
dest = Path('models/cm/aasist/last.pt')
dest.parent.mkdir(parents=True, exist_ok=True)

RESUMING = False
if hits:
    shutil.copy(hits[0], dest)
    RESUMING = True
    print(f'Restored checkpoint: {hits[0]} -> {dest}')
    if len(hits) > 1:
        print(f'({len(hits)} candidates found, used the first: {hits})')
else:
    print('No previous checkpoint found -- this is a fresh run (expected for run #1).')

## 4. Smoke test (skipped automatically if resuming)

Two epochs on a tiny subsample -- confirms the whole pipeline (data loading, the
front-end, augmentation, the model, checkpointing) actually runs on this environment
before committing real session time to a long run. Only meaningful on a fresh run --
if `RESUMING` is `True`, the pipeline already proved itself in an earlier session, and
running this with `--fresh` here would overwrite the checkpoint just restored above.

In [ ]:
if RESUMING:
    print('Resuming -- skipping the smoke test (already verified in an earlier session).')
else:
    !python scripts/train_cm.py --model aasist --epochs 2 --limit-per-class 500 --workers 2 --fresh

## 5. The real run

40 epochs over the full training set. This is the long-running cell -- when the session
ends (whether it finishes or hits the time cap), `last.pt` already has every epoch
completed so far checkpointed to it. **Do not add `--fresh` here** -- that would discard
either the smoke test's or a resumed run's progress. No `--limit-per-class` either: this
cell trains on the full set.

In [ ]:
!python scripts/train_cm.py --model aasist --epochs 40 --batch-size 16 --workers 2

## 6. Evaluate the result

Writes a new row into `results/tables/cm_aasist_cross_dataset.csv`, directly comparable
to the existing published-weights baseline (35.8% In-the-Wild EER) in the same-shaped
file this repo already ships. Safe to run even on a checkpoint that hasn't finished all
40 epochs yet -- `best.pt` always holds whatever epoch had the lowest dev EER so far, so
this gives an honest read on partial progress too, not only a finished run.

In [ ]:
!python scripts/eval_cm.py --model aasist

In [ ]:
import torch

ckpt = torch.load('models/cm/aasist/best.pt', map_location='cpu')
print(f"best checkpoint: epoch {ckpt['epoch']}, dev EER {ckpt['dev_eer']*100:.2f}%")
print(f"front_end fingerprint: {ckpt['front_end']}")
print()
print('Download this file from the notebook Output panel and place it locally at:')
print('  models/cm/aasist/best.pt')
print('then run (locally): python scripts/eval_cm.py --model aasist')

## Running this across multiple sessions

1. **Save Version -> Save & Run All (Commit)**, not the interactive Run button -- this is
   what lets it run unattended (see the note at the top). Give it a moment to queue and
   execute; you can close the tab once it's queued.
2. When it stops -- either it finished all 40 epochs, or it hit the session time cap --
   open that version's **Output** tab. `models/cm/aasist/last.pt` (to keep resuming) and
   `best.pt` (the actual result so far) are both there, same as every other file under
   `/kaggle/working/` at the end of the run.
3. **If epoch 40 isn't done yet:** commit a new version of this same notebook. Before
   running it, **Add Input -> Notebooks -> this notebook -> the version you just
   produced**, so cell 3 above ("Resuming a previous run") can find its `last.pt`. Save
   Version again. Repeat until training finishes.
4. **Once it's done:** download `best.pt` from the final version's Output, place it
   locally at `models/cm/aasist/best.pt` in the repo (or pass `--ckpt <path>` to
   `scripts/eval_cm.py` to point at it anywhere), and run
   `python scripts/eval_cm.py --model aasist` locally for real numbers against the
   published-weights baseline. `python scripts/serve_demo.py --ours` demos with it.